[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/spatialft/spatialft.github.io/blob/main/notebooks/02_dataset_prep.ipynb)

# Notebook 2 — Dataset Preparation

Download StepGame, split into train/eval, format for Unsloth fine-tuning.

In [ ]:
import os, sys
REPO = '/content/spatialft.github.io'
if not os.path.exists(REPO):
    !git clone https://github.com/spatialft/spatialft.github.io.git {REPO}
os.chdir(f'{REPO}/notebooks')
if REPO not in sys.path:
    sys.path.insert(0, REPO)


In [ ]:
import json
import random
from pathlib import Path

from src.dataset import load_stepgame, format_for_training

## Download StepGame

StepGame is available on HuggingFace datasets or from the original repo.
Run the cell below once to download raw splits.

In [ ]:
from datasets import load_dataset

ds = load_dataset("ZhengyanShi/StepGame")
print(ds)


In [ ]:
# Inspect a sample
print(ds['train'][0])

## Adapt field names if needed

StepGame fields vary by source. Adjust `story_field`, `question_field`, `answer_field` below.

In [ ]:
# ZhengyanShi/StepGame field mapping
STORY_FIELD    = "story"
QUESTION_FIELD = "question"
ANSWER_FIELD   = "label"
K_FIELD        = "k_hop"

TRAIN_SIZE     = 4000
EVAL_PER_K     = 50   # 50 examples per hop level → 500 total across k=1..10
SEED           = 42

def convert(ex):
    return {
        "story":    " ".join(ex[STORY_FIELD]) if isinstance(ex[STORY_FIELD], list) else ex[STORY_FIELD],
        "question": ex[QUESTION_FIELD],
        "answer":   ex[ANSWER_FIELD],
        "k":        ex[K_FIELD],
    }

# Train: first TRAIN_SIZE from train split
train_data = [convert(ex) for ex in ds["train"].select(range(TRAIN_SIZE))]

# Eval: stratified sample — EVAL_PER_K examples per k level
from collections import defaultdict
import random
random.seed(SEED)

by_k = defaultdict(list)
for ex in ds["validation"]:
    by_k[ex[K_FIELD]].append(ex)

eval_data = []
for k in sorted(by_k):
    sample = random.sample(by_k[k], min(EVAL_PER_K, len(by_k[k])))
    eval_data.extend(convert(ex) for ex in sample)

print(f"Train: {len(train_data)}")
print(f"Eval:  {len(eval_data)} ({len(by_k)} k-levels × up to {EVAL_PER_K} each)")
for k in sorted(by_k):
    n = sum(1 for ex in eval_data if ex["k"] == k)
    print(f"  k={k}: {n} examples")


In [ ]:
# Format for fine-tuning
train_formatted = [format_for_training(ex) for ex in train_data]

# Save
Path('../data/raw').mkdir(parents=True, exist_ok=True)
Path('../data/processed').mkdir(parents=True, exist_ok=True)
Path('../data/eval').mkdir(parents=True, exist_ok=True)

with open('../data/raw/train.json', 'w') as f:
    json.dump(train_data, f, indent=2)

with open('../data/processed/train_formatted.json', 'w') as f:
    json.dump(train_formatted, f, indent=2)

with open('../data/eval/stepgame_eval.json', 'w') as f:
    json.dump(eval_data, f, indent=2)

print('Saved.')
print('Sample formatted:')
print(train_formatted[0]['full_text'][:500])